### **1. Prepare the environment**

<font size=2>Using the magic `!` command, create the `dbs` and `output` directories with the Bash `mkdir` command. Just provide all paths/directory names separated by spaces after entering the command itself. You can use the `-p` parameter to avoid the error from appearing when directories already exist (e.g. when rerunning the entire notebook code).
</font>

In [1]:
!mkdir -p dbs output

A subdirectory or file output already exists.
Error occurred while processing: output.


### **2. BLAST: Creating a Database**


#### _makeblastdb_ tool and database creation using redirection to _stdin_

<font size=2>Knowing that in the Bash text shell:

* standard output (<i>standard output</i>, <i>stdout</i>) is what you see on the screen by default
* _stdout_ from one application can be redirected to standard input (<i>standard input</i>, <i>stdin</i>) to another with `|`,\
e.g.: `app1 | app2`
* application `cat file1.txt file2.txt ...` copies the contents of the file(s) to <i>stdout</i>
* the names of multiple files with the same name pattern can be replaced with a replacement expression (<i>wildcard</i>),\
e.g.: `cat *.txt`
* to create a nucleotide database with the `makeblastdb` tool, you must provide it with at least four parameters: `-in` the path to the fasta sequence file, `-dbtype` <i>nucl</i> in the case of nucleotide sequences, `-title` your own database name and `-out` the path to the directory with the prefix of the file names of the database being created
* the `-in` parameter accepts only one file, you can omit it and then `makeblastdb` will read data from the <i>stdin</i> stream

and using the appropriate `%%magic command`, enter and run the Bash code in the cell below, which will use the `cat` application to drop the contents of `*.fna` files from the `genomes` directory to <i>stdout</i> and redirect them to <i>stdin</i> `makeblastdb` to create the `genomes` nucleotide database in the `dbs` directory
\
\
Then add the line `ls -l dbs` which will display the contents of the `dbs` directory

</font>

In [3]:
%%bash
cat genomes/*.fna | makeblastdb -dbtype nucl -title "genomes" -out dbs/genomes
ls -l dbs



Building a new DB, current time: 10/23/2025 16:03:52
New DB name:   /Users/aleksanderstanuch/dev/bioinformatics/cwiczenia2/dbs/genomes
New DB title:  genomes
Sequence type: Nucleotide
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 222 sequences in 0.653294 seconds.


total 13808
-rw-r--r--@ 1 aleksanderstanuch  staff    81920 Oct 23 16:03 genomes.ndb
-rw-r--r--@ 1 aleksanderstanuch  staff    43232 Oct 23 16:03 genomes.nhr
-rw-r--r--@ 1 aleksanderstanuch  staff     2756 Oct 23 16:03 genomes.nin
-rw-r--r--@ 1 aleksanderstanuch  staff      454 Oct 23 16:03 genomes.njs
-rw-r--r--@ 1 aleksanderstanuch  staff     2672 Oct 23 16:03 genomes.not
-rw-r--r--@ 1 aleksanderstanuch  staff  6857061 Oct 23 16:03 genomes.nsq
-rw-r--r--@ 1 aleksanderstanuch  staff    65536 Oct 23 16:03 genomes.ntf
-rw-r--r--@ 1 aleksanderstanuch  staff      892 Oct 23 16:03 genomes.nto


### **3. BLAST: database search**
<font size=2>Then, in the next cell, enter and run the `blastn` tool, specifying as the _query_ sequence the path to the `input/gap.fna` file, as the database, the path to the one you just created, and save the results to the `output/blastn.txt` file. Don't forget about the appropriate `%%magic command` which will allow you to run the contents of the entire cell as Bash code.
\
\
Then add the line `head -25 output/blast.txt`, which will display the first 25 lines of the results file.
\
\
Finally, go to the Jupyter home screen, go to the `output` directory, open and view the entire contents of the `blastn.txt` file.</font>

In [6]:
%%bash
blastn -query input/gap.fna -db dbs/genomes -out output/blastn.txt
head -25 output/blastn.txt

BLASTN 2.17.0+


Reference: Zheng Zhang, Scott Schwartz, Lukas Wagner, and Webb
Miller (2000), "A greedy algorithm for aligning DNA sequences", J
Comput Biol 2000; 7(1-2):203-14.



Database: genomes
           222 sequences; 27,426,152 total letters



Query= gap

Length=1011
                                                                      Score     E
Sequences producing significant alignments:                          (Bits)  Value

UDMK01000001.1 Staphylococcus aureus strain EOE252 genome assembl...  1851    0.0  
UDIR01000001.1 Staphylococcus aureus strain EOE046 genome assembl...  1851    0.0  
UHCG01000001.1 Staphylococcus aureus strain NCTC9553 genome assem...  1851    0.0  
OGBR01000034.1 Staphylococcus aureus strain RH238 genome assembly...  1851    0.0  
FQKY01000001.1 Staphylococcus argenteus strain 3688STDY6125110 ge...  1757    0.0  


### **4. BLAST: receiving results in tabular format**
<font size=2>Copy the code from the previous cell and modify it so that the `blastn` tool writes the results in tabular form to the file `output/blast.tsv` with tab as a column separator and with `sseqid sstart send qcovs pident evalue` columns. What values ​​do these columns refer to? If you don't know, type `blastn -help` in a text terminal and check.
\
\
With the `head` command, display only the first 5 lines of the results file.
\
\
Finally, go to the Jupyter home screen, go to the `output` directory, open and view the entire contents of the `blastn.tsv` file.

In [8]:
%%bash
blastn -query input/gap.fna -db dbs/genomes -out output/blast.tsv -outfmt "6 sseqid sstart send qcovs pident evalue"
head -5 output/blast.tsv


UDMK01000001.1	276652	277662	100	99.703	0.0
UDIR01000001.1	129950	128940	100	99.703	0.0
UHCG01000001.1	795488	796498	100	99.703	0.0
OGBR01000034.1	321008	322018	100	99.703	0.0
FQKY01000001.1	1106566	1105556	100	98.022	0.0


### **5. Polars: processing BLAST results**

<font size=2>To complete the next steps, use the presentation from the last lecture as a guide.</font>

In [9]:
# import the polars module with the name 'pl'
import polars as pl

In [11]:
# load results from the output/blastn.tsv file into a DataFrame object
# and then display the table
df = pl.read_csv("output/blast.tsv", separator="\t", has_header=False)
df.columns = ["sseqid", "sstart", "send", "qcovs", "pident", "evalue"]
df

sseqid,sstart,send,qcovs,pident,evalue
str,i64,i64,i64,f64,f64
"""UDMK01000001.1""",276652,277662,100,99.703,0.0
"""UDIR01000001.1""",129950,128940,100,99.703,0.0
"""UHCG01000001.1""",795488,796498,100,99.703,0.0
"""OGBR01000034.1""",321008,322018,100,99.703,0.0
"""FQKY01000001.1""",1106566,1105556,100,98.022,0.0
"""FQRR01000001.1""",1121207,1120197,100,97.923,0.0
"""AE015929.1""",553355,554365,100,89.229,0.0
"""FKIN01000005.1""",1151570,1150579,98,86.847,0.0
"""LR134263.1""",2257955,2256964,98,86.231,0.0


In [12]:
# create a new table by keeping from the output table
# rows where the value of the qcovs column is 100,
# display it
df2 = df.filter(pl.col("qcovs") == 100)
df2


sseqid,sstart,send,qcovs,pident,evalue
str,i64,i64,i64,f64,f64
"""UDMK01000001.1""",276652,277662,100,99.703,0.0
"""UDIR01000001.1""",129950,128940,100,99.703,0.0
"""UHCG01000001.1""",795488,796498,100,99.703,0.0
"""OGBR01000034.1""",321008,322018,100,99.703,0.0
"""FQKY01000001.1""",1106566,1105556,100,98.022,0.0
"""FQRR01000001.1""",1121207,1120197,100,97.923,0.0
"""AE015929.1""",553355,554365,100,89.229,0.0


In [ ]:
# save the table with filtered results
# to the output/filtered.tsv file
# and then from the Jupyter Notebook home screen
# go to the output directory and browse the saved files
# results and compare them with the contents of the z file
# output results

pl.DataFrame.write_csv(df2, "output/filtered.tsv", separator="\t")

### **6. Polars: '_DataFrame_' in text mode**

<font size=2>Start a text terminal, change to the current working directory with the Bash command `cd directory_path`, run the `ipython` text interpreter, load the table with the output results from the `blastn.tsv` file into the DataFrame structure and browse it in text mode.
\
\
If you don't remember the path of the current working directory, use the magic command `%pwd`.</font>